# 12 — CIFAR-10N / CIFAR-100N: group-level conditions (independent-dataset test)

**What this shows.** On AlleNoise (K=5691) no group term beat a trivial per-item aggregation
(nb06), and nb08 argued — by shrinking the label space *internally* — that the group signal
should recover as K drops. A reviewer can object that an internal subset of one dataset is weak
evidence. This notebook supplies the **independent** test: two datasets with **real human label
noise** and genuinely small label spaces, **K=10 and K=100**, with their own images, backbone,
optimizer, and noise process.

**Design.** CIFAR-N has no taxonomy, so the natural entity grouping is the **noisy class** itself:
each registered class is one "group", and its ground-truth noise rate is the fraction of its
members whose clean label differs. We compute, per group, the same scores as nb06 — the best
baselines (mean / p90 of per-item mismatch, mean entropy) and the GUARD contamination terms
(C_proj and C_proj·kappa) — and rank groups by true noise rate. Because K is small, the subtree
projection is over the full simplex (the "subtree" is all classes), so C_proj = C_full here; we
report it under both names for continuity. We use the **same metrics** (Spearman, AUROC of
high- vs low-noise groups) and the **same paired bootstrap** as nb07.

**Reads only** the artifacts nb11 saved. No retraining. Run nb11 for BOTH datasets first.


In [1]:
# Notebook: 12_cifar_group_conditions  (reads nb11 artifacts only; self-contained helpers)
import os, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["axes.edgecolor"]="0.2"; plt.rcParams["axes.linewidth"]=0.8
plt.rcParams["font.family"]="DejaVu Sans"
GREYS=["#111111","#555555","#888888","#bbbbbb","#dddddd"]
FIG=os.path.join("..","results","figures"); TAB=os.path.join("..","results","tables")
os.makedirs(FIG,exist_ok=True); os.makedirs(TAB,exist_ok=True)
def savefig(fig,name):
    for ext in ("png","pdf"): fig.savefig(os.path.join(FIG,f"{name}.{ext}"),dpi=600,bbox_inches="tight")
    plt.close(fig)

EPS=1e-12
def entropy(p,axis=-1):
    p=np.clip(p,EPS,1.0); return -np.sum(p*np.log(p),axis=axis)
def js_divergence(q,p):
    q=np.clip(np.asarray(q,float),EPS,1.0); p=np.clip(np.asarray(p,float),EPS,1.0)
    m=0.5*(q+p)
    return float(0.5*np.sum(q*np.log2(q/m))+0.5*np.sum(p*np.log2(p/m)))

In [2]:
# --- per-dataset group analysis: group = noisy class ---
def analyze_cifar(dataset):
    D=os.path.join("..","data","cifar",dataset)
    meta=pd.read_parquet(os.path.join(D,"item_meta.parquet"))
    pi=np.load(os.path.join(D,"pi_memmap.npy"),mmap_mode="r"); N,K=pi.shape
    noisy=meta["noisy_id"].values; mis=meta["is_misregistered"].values
    p_noisy=meta["p_noisy"].values; ent=meta["entropy"].values
    s_ind=1.0-p_noisy                                   # per-item mismatch signal

    recs=[]
    for g in range(K):
        idx=np.where(noisy==g)[0]
        if len(idx)<30: continue
        pbar=np.asarray(pi[idx],dtype=np.float64).mean(0)      # mean belief of the group
        q=np.zeros(K); q[g]=1.0                                # group claims its registered class
        # projection onto the group's own support is trivial here (single claimed class),
        # so we evaluate C over the full simplex: C_proj == C_full for CIFAR (K small).
        C=js_divergence(q,pbar)
        # direction concentration kappa: how peaked the positive residual (q-pbar)+ is
        r=np.clip(q-pbar,0,None); ssum=r.sum()
        kap=0.0 if ssum<=1e-12 else 1-entropy(r/ssum)/np.log(K)
        recs.append(dict(dataset=dataset,group=g,n=len(idx),
            true_noise=float(mis[idx].mean()),
            mean_1mp=float(s_ind[idx].mean()),
            p90_1mp=float(np.quantile(s_ind[idx],0.9)),
            mean_ent=float(ent[idx].mean()),
            C_full=C, C_proj=C, out_mass=0.0, Cproj_kappa=C*kap))
    g=pd.DataFrame(recs)
    return g,N,K

frames={}
for ds in ["cifar10n","cifar100n"]:
    try:
        g,N,K=analyze_cifar(ds); frames[ds]=g
        print(f"{ds}: N={N:,} K={K}  groups={len(g)}  mean true noise={g.true_noise.mean():.3%}")
    except FileNotFoundError:
        print(f"[skip] {ds}: run nb11 with DATASET='{ds}' first.")

cifar10n: N=50,000 K=10  groups=10  mean true noise=8.807%
cifar100n: N=50,000 K=100  groups=100  mean true noise=38.915%


In [3]:
# --- scoring: Spearman + AUROC(high vs low noise groups), same protocol as nb06/07 ---
def score_table(g):
    cols=["mean_1mp","p90_1mp","mean_ent","C_full","C_proj","out_mass","Cproj_kappa"]
    lo,hi=g.true_noise.quantile(1/3),g.true_noise.quantile(2/3)
    sub=g[(g.true_noise<=lo)|(g.true_noise>=hi)].copy()
    y=(sub.true_noise>=hi).astype(int).values
    rows=[]
    for c in cols:
        rho=spearmanr(g[c],g.true_noise)[0]
        au=roc_auc_score(y,sub[c].values); au=max(au,1-au)
        rows.append(dict(score=c,
            kind=("baseline" if c in ("mean_1mp","p90_1mp","mean_ent") else "GUARD"),
            spearman=round(float(rho),3), auroc_hi_lo=round(float(au),3)))
    return pd.DataFrame(rows)

for ds,g in frames.items():
    comp=score_table(g)
    comp.insert(0,"dataset",ds)
    comp.to_csv(os.path.join(TAB,f"t12_{ds}_group_comparison.csv"),index=False)
    g.to_csv(os.path.join(TAB,f"t12_{ds}_groups.csv"),index=False)
    print(f"\n=== {ds} (K={'10' if ds=='cifar10n' else '100'}) group-score comparison ===")
    print(comp.to_string(index=False))


=== cifar10n (K=10) group-score comparison ===
 dataset       score     kind  spearman  auroc_hi_lo
cifar10n    mean_1mp baseline     0.382        0.833
cifar10n     p90_1mp baseline     0.794        1.000
cifar10n    mean_ent baseline    -0.006        0.583
cifar10n      C_full    GUARD     0.382        0.833
cifar10n      C_proj    GUARD     0.382        0.833
cifar10n    out_mass    GUARD       NaN        0.500
cifar10n Cproj_kappa    GUARD     0.382        0.833

=== cifar100n (K=100) group-score comparison ===
  dataset       score     kind  spearman  auroc_hi_lo
cifar100n    mean_1mp baseline     0.745        0.938
cifar100n     p90_1mp baseline     0.342        0.735
cifar100n    mean_ent baseline     0.576        0.843
cifar100n      C_full    GUARD     0.745        0.938
cifar100n      C_proj    GUARD     0.745        0.938
cifar100n    out_mass    GUARD       NaN        0.500
cifar100n Cproj_kappa    GUARD     0.745        0.938


C:\Users\miy\AppData\Local\Temp\ipykernel_36000\372994809.py:9: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho=spearmanr(g[c],g.true_noise)[0]
C:\Users\miy\AppData\Local\Temp\ipykernel_36000\372994809.py:9: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho=spearmanr(g[c],g.true_noise)[0]


In [4]:
# --- paired bootstrap: GUARD vs the BEST baseline, with a FIXED hi/lo split (same test as nb07) ---
# Fix (reviewer-proofing): compare against the strongest baseline, not a hard-coded one, and keep
# the tertile split FIXED while resampling groups -- otherwise the split is recomputed inside each
# bootstrap draw and the delta becomes unstable / meaningless on small group counts.
def paired_bootstrap(g, a="C_proj", B=5000, seed=0):
    rng=np.random.default_rng(seed)
    lo,hi=g.true_noise.quantile(1/3),g.true_noise.quantile(2/3)
    sub=g[(g.true_noise<=lo)|(g.true_noise>=hi)].reset_index(drop=True)
    y=(sub.true_noise>=hi).astype(int).values
    def au(s):
        v=roc_auc_score(y,s); return max(v,1-v)
    # pick the best baseline by point AUROC on the fixed split
    bases=["mean_1mp","p90_1mp","mean_ent"]
    b=max(bases, key=lambda c: au(sub[c].values))
    idx=np.arange(len(sub)); diffs=[]
    av=sub[a].values; bv=sub[b].values
    for _ in range(B):
        bs=rng.choice(idx,len(idx),replace=True); yy=y[bs]
        if yy.min()==yy.max(): continue
        # AUROC with the FIXED labels y (split not recomputed)
        va=roc_auc_score(yy,av[bs]); va=max(va,1-va)
        vb=roc_auc_score(yy,bv[bs]); vb=max(vb,1-vb)
        diffs.append(va-vb)
    diffs=np.array(diffs)
    return dict(score_a=a,best_baseline=b,
                auroc_a=round(au(av),3),auroc_b=round(au(bv),3),
                delta=round(float(diffs.mean()),3),
                lo=round(float(np.quantile(diffs,0.025)),3),
                hi=round(float(np.quantile(diffs,0.975)),3),
                p_gt0=round(float((diffs>0).mean()),3))

rows=[]
for ds,g in frames.items():
    r=paired_bootstrap(g); r["dataset"]=ds; rows.append(r)
    print(f"{ds}: GUARD(C_proj) AUROC={r['auroc_a']} vs best baseline '{r['best_baseline']}'={r['auroc_b']}  "
          f"delta={r['delta']} [{r['lo']}, {r['hi']}]  P(delta>0)={r['p_gt0']}")
if rows:
    pd.DataFrame(rows)[["dataset","score_a","best_baseline","auroc_a","auroc_b","delta","lo","hi","p_gt0"]]\
        .to_csv(os.path.join(TAB,"t12_paired_bootstrap.csv"),index=False)
    print("\nsaved t12_paired_bootstrap.csv (vs best baseline, fixed split)")

cifar10n: GUARD(C_proj) AUROC=0.833 vs best baseline 'p90_1mp'=1.0  delta=-0.148 [-0.5, 0.0]  P(delta>0)=0.001
cifar100n: GUARD(C_proj) AUROC=0.938 vs best baseline 'mean_1mp'=0.938  delta=0.0 [0.0, 0.0]  P(delta>0)=0.0

saved t12_paired_bootstrap.csv (vs best baseline, fixed split)


In [5]:
# --- figure: AUROC by score, CIFAR-10N vs CIFAR-100N vs AlleNoise(K=5691) ---
# Shows ALL baselines (incl. mean per-example mismatch) so the best baseline is visible and the
# reader can see GUARD matches, not beats, it. The best baseline per panel is hatched.
def load_allenoise():
    p=os.path.join(TAB,"t06_comparison_summary_d4.csv")
    if not os.path.exists(p): return None
    d=pd.read_csv(p); return dict(zip(d.score,d.auroc_hi_lo))

alle=load_allenoise()
panels=[("cifar10n","CIFAR-10N (K=10)"),("cifar100n","CIFAR-100N (K=100)")]
show=["mean_1mp","p90_1mp","mean_ent","C_proj","Cproj_kappa"]   # mean_1mp now included
labels={"mean_1mp":"mean (base)","p90_1mp":"p90 (base)","mean_ent":"entropy (base)",
        "C_proj":"C_proj (GUARD)","Cproj_kappa":"C_proj·κ (GUARD)"}
BASES=("mean_1mp","p90_1mp","mean_ent")

def draw(ax, getval, title):
    vals=[getval(s) for s in show]
    colors=[GREYS[3] if s in BASES else GREYS[0] for s in show]
    bars=ax.bar(range(len(show)),vals,color=colors,edgecolor="black",linewidth=0.6)
    # hatch the single best baseline so "GUARD == best baseline" is legible
    base_vals={s:getval(s) for s in BASES}
    best=max(base_vals,key=base_vals.get)
    bars[show.index(best)].set_hatch("////")
    ax.axhline(0.5,color="black",ls=":",lw=1)
    ax.set_xticks(range(len(show))); ax.set_xticklabels([labels[s] for s in show],rotation=35,ha="right")
    ax.set_title(title); ax.set_ylim(0.4,1.0)

ncol=len(panels)+(1 if alle else 0)
fig,axes=plt.subplots(1,ncol,figsize=(3.6*ncol,3.4),sharey=True)
if ncol==1: axes=[axes]
for ax,(ds,title) in zip(axes,panels):
    if ds not in frames: ax.set_visible(False); continue
    comp=score_table(frames[ds]).set_index("score")
    draw(ax, lambda s: comp.loc[s,"auroc_hi_lo"], title)
axes[0].set_ylabel("AUROC (high vs low noise group)")
if alle:
    draw(axes[-1], lambda s: alle.get(s,np.nan), "AlleNoise (K=5691)")
# legend: hatch = best baseline
from matplotlib.patches import Patch
axes[-1].legend(handles=[Patch(facecolor=GREYS[3],hatch="////",edgecolor="black",label="best baseline")],
                frameon=False, loc="upper right", fontsize=8)
savefig(fig,"f12_cifar_conditions")
print("saved f12_cifar_conditions.{png,pdf} (now includes mean baseline; best baseline hatched)")

C:\Users\miy\AppData\Local\Temp\ipykernel_36000\372994809.py:9: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho=spearmanr(g[c],g.true_noise)[0]
C:\Users\miy\AppData\Local\Temp\ipykernel_36000\372994809.py:9: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho=spearmanr(g[c],g.true_noise)[0]


saved f12_cifar_conditions.{png,pdf} (now includes mean baseline; best baseline hatched)


## How to read the result (for the paper)

- If, on **CIFAR-10N (K=10)** and **CIFAR-100N (K=100)**, the GUARD contamination term
  (`C_proj` / `C_proj·κ`) reaches a clearly higher group-ranking AUROC than the per-item
  baselines — and the paired bootstrap `delta` CI in `t12_paired_bootstrap.csv` excludes 0 — then
  the paper's predicted condition is confirmed **on independent data**: group-level uncertainty
  *is* informative once the label space is small and beliefs are sharp. That turns Contribution 2
  from a claim into a demonstration and directly answers the "single dataset" review.
- If the gap is positive but the CI still straddles 0, report it as *consistent with* the
  predicted trend, alongside the AlleNoise internal K-sweep (nb08), rather than as proof.
- Tables written: `t12_<ds>_groups.csv`, `t12_<ds>_group_comparison.csv`,
  `t12_paired_bootstrap.csv`. Figure: `f12_cifar_conditions.{png,pdf}`.

Everything above reads only nb11's saved artifacts; no retraining happens here.
